In [1]:
# 风险及免责提示：该策略由聚宽用户在聚宽社区分享，仅供学习交流使用。
# 原文一般包含策略说明，如有疑问请到原文和作者交流讨论。
# 原文网址：https://www.joinquant.com/view/community/detail/37280
# 标题：可转债双低策略
# 作者：未来道士

from jqdata import bond
import numpy as np
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from pylab import mpl
import seaborn
import os
seaborn.set()
mpl.rcParams['font.sans-serif'] = ['KaiTi']
mpl.rcParams['axes.unicode_minus'] = False
from jqdata import finance
from jqdata import get_all_trade_days
import pandas as pd

In [2]:
import datetime
import re
from bs4 import BeautifulSoup
import requests

In [3]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker
plt.style.use('seaborn-white')

In [4]:
pd.set_option('display.max_rows', 300)

In [7]:
# 爬虫相关函数，用于从集思录获取可转债评级和到期赎回价格信息
# 由于聚宽无法拿到可转债赎回价格和评级数据，只能通过爬集思录获得。。。
# 后续聚宽数据完善后可以删除集思录爬虫部分
def getHTMLtext(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/72.0.3626.119 Safari/537.36"
    }
    r = requests.get(url=url, headers=headers)
    r.raise_for_status()
    r.encoding=r.apparent_encoding
    return r.text

def get_bond_extra_info(code):
    url = 'https://www.jisilu.cn/data/convert_bond_detail/' + code
    html=getHTMLtext(url)
    soup=BeautifulSoup(html,'html.parser')
    price = float(soup.find('td', id = 'redeem_price').get_text().strip())
    rating = soup.find('td', id = 'rating_cd').get_text().strip()
    return price, rating

def get_before_after_trade_days(date, count, is_before=True):
    """
    date :查询日期
    count : 前后追朔的数量
    is_before : True , 前count个交易日  ; False ,后count个交易日

    返回 : 基于date的日期, 向前或者向后count个交易日的日期 ,一个datetime.date 对象
    """
    all_date = pd.Series(get_all_trade_days())
    if isinstance(date,str):
        all_date = all_date.astype(str)
    if isinstance(date,datetime.datetime):
        date = date.date()
    print(all_date.tail(10))

    if is_before :
        return all_date[all_date< date].tail(count).values[0]
    else :
        return all_date[all_date>date].head(count).values[-1]

#获得具体某只可转债指定日期可转债价格，股票价格，行权价格双低值，是否到期，是否可交易等信息
def get_bond_detail(code, date):
    today_date = datetime.datetime.strptime(str(date), "%Y-%m-%d")
    today = today_date.strftime("%Y-%m-%d")
    date_1_y = today_date + datetime.timedelta(days=-365)
    one_year_ago = date_1_y.strftime("%Y-%m-%d")
    df=bond.run_query(query(bond.BOND_BASIC_INFO).filter(bond.BOND_BASIC_INFO.bond_type_id== '703013',
                                                     bond.BOND_BASIC_INFO.code==code).limit(1000))
    conv_df = bond.run_query(query(bond.CONBOND_BASIC_INFO).filter(bond.CONBOND_BASIC_INFO.code==code).limit(100))
    bond_name = df.loc[0, 'short_name']
    stock_code = df.loc[0, 'company_code']
    mature_date = df.loc[0, 'maturity_date']
    bond_start_date = df.loc[0, 'list_date']
    # 以下代码用于判断可转债是否可交易，是否终止上市
    if(not pd.isna(bond_start_date)):
        bond_start_date_str = bond_start_date.strftime('%Y-%m-%d')
    else:
        bond_start_date = df.loc[0, 'interest_begin_date']
        bond_start_date_str = bond_start_date.strftime('%Y-%m-%d')
    conv_start_date = conv_df.loc[0, 'convert_start_date']
    conv_start_date_str = conv_start_date.strftime('%Y-%m-%d')
    mature_date_str = mature_date.strftime('%Y-%m-%d')
    if(today <= bond_start_date_str):
        print('date:{}|bond:{}|invalid'.format(today, code))
        return np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, 0, 0, 0
    conv_bond_price_df = bond.run_query(query(bond.CONBOND_DAILY_PRICE).filter(bond.CONBOND_DAILY_PRICE.date<today,
                                                         bond.CONBOND_DAILY_PRICE.code==code).limit(1000))
    if(len(conv_bond_price_df)==0):
        print('date:{}|bond:{}|invalid'.format(today, code))
        return np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, 0, 0, 0
    if(today > mature_date_str):
        print('date:{}|bond:{}|stop trading'.format(today, code))
        return np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, 0, 1, 1
    if(today > conv_start_date_str):
        conv_info = bond.run_query(query(bond.CONBOND_DAILY_CONVERT).filter(bond.CONBOND_DAILY_CONVERT.code==code,
                                                       bond.CONBOND_DAILY_CONVERT.date>=today).limit(100))
        if(len(conv_info)==0):
            print('date:{}|bond:{}|stop trading'.format(today, code))
            return np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, np.NaN, 0, 1, 1
    # 获得对应正股价格信息
    price_df = get_price(security=stock_code, start_date=one_year_ago, end_date=today, frequency='daily', fields=None, skip_paused=False, fq=None, panel=True)
    stock_price = price_df.tail(1).reset_index(drop=True).loc[0, 'close']
    stock_date = price_df.index[len(price_df.index)-1].strftime("%Y-%m-%d")
    
    price_change = bond.run_query(query(bond.CONBOND_CONVERT_PRICE_ADJUST).filter(bond.CONBOND_CONVERT_PRICE_ADJUST.code==code,
                                                         ).limit(1000))
    strike_price = price_change[price_change.adjust_date.map(lambda x:x.strftime("%Y-%m-%d")<=stock_date)].tail(1).reset_index(drop=True).loc[0, 'new_convert_price']
    cur_price = price_df.tail(1).reset_index(drop=True).loc[0, 'close']
    
    #获得可转债价格，用于计算双低值
    bond_price_df = bond.run_query(query(bond.CONBOND_DAILY_PRICE).filter(bond.CONBOND_DAILY_PRICE.date== stock_date,
                                                         bond.CONBOND_DAILY_PRICE.code==code).limit(1000))
    bond_price = bond_price_df.tail(1).reset_index(drop=True).loc[0, 'close']
    # 溢价率，双低计算
    final_price, rating = get_bond_extra_info(code)
    bond_inner_value = 100/strike_price*cur_price
    double_low = bond_price + (bond_price-bond_inner_value)/bond_inner_value*100
    return 0, bond_price, stock_price, strike_price, 0, 0, double_low, 1, 0, 1

def get_bond_info(date):
    file_name = 'conv_bond_cache/conv_bond_info_' + date
    if(os.path.exists(file_name)):
        result = pd.read_pickle(file_name)
        return result
    #bond_df=bond.run_query(query(bond.BOND_BASIC_INFO).filter(bond.BOND_BASIC_INFO.bond_type_id== '703013',
    #                                                     bond.BOND_BASIC_INFO.list_status_id == '301001').limit(1000))
    bond_df=bond.run_query(query(bond.BOND_BASIC_INFO).filter(bond.BOND_BASIC_INFO.bond_type_id== '703013').limit(2000))
    bond_df=bond_df[bond_df.maturity_date.map(lambda x:x.strftime("%Y-%m-%d")>date)]
    
    bond_theo_lst = []
    bond_price_lst = []
    stock_price_lst = []
    strike_price_lst = []
    deviation_lst = []
    company_type_lst = []
    premium_rate_lst = []
    double_low_lst = []
    score_lst = []
    tradeable_lst = []
    reach_first_trade_day_lst=[]
    is_delist_lst = []
    print('total_df_len:{}'.format(len(bond_df)))
    i=0
    exception_num = 0
    for info_row in bond_df.iterrows():
        code = info_row[1].code
        security_code = info_row[1].company_code
        industry_val = get_industry(security_code)
        industry_type = industry_val[security_code]['sw_l1']['industry_name']
        print('process:{}|percentage:{}'.format(code, i/len(bond_df)))
        is_delist = 0
        reach_first_trade_day = 0
        i=i+1
        '''
        bond_theo, bond_price, stock_price, strike_price, deviation, premium_rate, double_low, tradeable, is_delist, reach_first_trade_day = get_bond_detail(code, date)
        deviation_rate = deviation*100
        score = double_low + deviation_rate
        '''
        tradeable=0
        try:
            bond_theo, bond_price, stock_price, strike_price, deviation, premium_rate, double_low, tradeable, is_delist, reach_first_trade_day = get_bond_detail(code, date)
            deviation_rate = deviation*100
            score = double_low + deviation_rate
        
        except:
            exception_num = exception_num+1
            print('exception exist for code:{}|except_total_num:{}'.format(code, exception_num))
            bond_theo=np.NaN
            bond_price=np.NaN
            stock_price = np.NaN
            strike_price = np.NaN
            deviation=np.NaN
            deviation_rate=np.NaN
            premium_rate=np.NaN
            double_low = np.NaN
            score = np.NaN
            tradeable = 0
            is_delist = 0
            reach_first_trade_day = 1
        
        
        bond_theo_lst.append(bond_theo)
        bond_price_lst.append(bond_price)
        stock_price_lst.append(stock_price)
        strike_price_lst.append(strike_price)
        deviation_lst.append(deviation_rate)
        premium_rate_lst.append(premium_rate)
        double_low_lst.append(double_low)
        score_lst.append(score)
        company_type_lst.append(industry_type)
        tradeable_lst.append(tradeable)
        is_delist_lst.append(is_delist)
        reach_first_trade_day_lst.append(reach_first_trade_day)
    bond_df_new = bond_df.assign(bond_theo=bond_theo_lst, bond_price=bond_price_lst, stock_price = stock_price_lst, strike_price = strike_price_lst, deviation=deviation_lst, company_type=company_type_lst, premium_rate=premium_rate_lst, double_low=double_low_lst, score=score_lst, tradeable=tradeable_lst, reach_first_trade_day=reach_first_trade_day_lst, is_delist=is_delist_lst)
    bond_df_reslt = bond_df_new
    bond_df_reslt.to_pickle(file_name)
    print('total except num:{}'.format(exception_num))
    return bond_df_reslt

In [8]:
# 以下为回测主逻辑，用于获得双低策略收益曲线
def get_target_conv_bond_by_double_low(date, candidate_num, money):
    conv_bond_info = get_bond_info(date)
    tradeable_bond = conv_bond_info[(conv_bond_info.tradeable==1)&(conv_bond_info.reach_first_trade_day==1)&(conv_bond_info.is_delist==0)&(conv_bond_info.bond_price>10)].reset_index(drop=True)
    target_conv_bond = tradeable_bond.sort_values(by='double_low').head(candidate_num).reset_index(drop=True)
    money_for_each_bond = money / candidate_num
    target_trade_result = []
    for conv_bond in target_conv_bond.iterrows():
        code = conv_bond[1].code
        bond_price = conv_bond[1].bond_price
        qty = money_for_each_bond / bond_price
        target_trade_result.append((code, bond_price, qty))
    return target_trade_result

def get_benchmark_conv_bond(date, candidate_num, money):
    conv_bond_info = get_bond_info(date)
    tradeable_bond = conv_bond_info[(conv_bond_info.tradeable==1)&(conv_bond_info.reach_first_trade_day==1)&(conv_bond_info.is_delist==0)&(conv_bond_info.bond_price>10)].reset_index(drop=True)
    bond_num = len(tradeable_bond)
    money_for_each_bond = money / bond_num
    target_trade_result = []
    for conv_bond in tradeable_bond.iterrows():
        code = conv_bond[1].code
        bond_price = conv_bond[1].bond_price
        qty = money_for_each_bond / bond_price
        target_trade_result.append((code, bond_price, qty))
    return target_trade_result
    
def sell_bond(date, hold_bond):
    conv_bond_info = get_bond_info(date)
    hold_bond_code_lst = []
    for val in hold_bond:
        hold_bond_code_lst.append(val[0])
    bond_to_sell_info = conv_bond_info[conv_bond_info.code.map(lambda x:x in hold_bond_code_lst)].reset_index(drop=True)
    money = 0
    for conv_bond in bond_to_sell_info.iterrows():
        code = conv_bond[1].code
        bond_price = conv_bond[1].bond_price
        tradeable = conv_bond[1].tradeable
        delist = conv_bond[1].is_delist
        if(delist):
            for val in hold_bond:
                if(val[0]==code):
                    money = money + val[1]*val[2]
                    hold_bond.remove(val)
                    break
        else:
            if((tradeable) and (bond_price>0)):
                for val in hold_bond:
                    if(val[0]==code):
                        money = money + bond_price*val[2]
                        hold_bond.remove(val)
                        break
    return money, hold_bond

def update_bond_price(date, hold_bond):
    conv_bond_info = get_bond_info(date)
    hold_bond_code_lst = []
    new_hold_bond = []
    for val in hold_bond:
        hold_bond_code_lst.append(val[0])
        new_hold_bond.append((val[0], val[1], val[2]))
    bond_to_calc = conv_bond_info[conv_bond_info.code.map(lambda x:x in hold_bond_code_lst)].reset_index(drop=True)
    for conv_bond in bond_to_calc.iterrows():
        code = conv_bond[1].code
        bond_price = conv_bond[1].bond_price
        tradeable = conv_bond[1].tradeable
        delist = conv_bond[1].is_delist
        if((tradeable) and (bond_price>0)):
            for val in new_hold_bond:
                if(val[0]==code):
                    new_val = (val[0], bond_price, val[2])
                    new_hold_bond.remove(val)
                    new_hold_bond.append(new_val)
                    break
    return new_hold_bond
        

def get_bond_total_val(holding_bond):
    total_val = 0
    for val in holding_bond:
        total_val = total_val + val[1]*val[2]
    return total_val

def backtest(start_date, trading_dates, select_target_func, money, candidate_bond_num, trade_interval=1):
    date = start_date
    cur_trade_date = get_before_after_trade_days(date, 1, is_before=False)
    left_trade_date = trading_dates
    result = []
    cur_money = money
    holding_bond = []
    interval = 0
    while(left_trade_date>0):
        #print('backtest|date:{}|left:{}'.format(cur_trade_date, left_trade_date))
        if(interval == 0):
            sell_money, remain_bond = sell_bond(cur_trade_date, holding_bond)
            cur_money = sell_money + cur_money
            target_trade_result = select_target_func(date=cur_trade_date, candidate_num=candidate_bond_num, money=cur_money)
            cur_money = 0
            holding_bond = remain_bond + target_trade_result
            interval = trade_interval-1
        else:
            holding_bond = update_bond_price(cur_trade_date, holding_bond)
            interval = interval-1
        total_value = get_bond_total_val(holding_bond)
        #print('date:{}|hold_bond:{}|value:{}'.format(cur_trade_date, holding_bond, total_value))
        result.append((cur_trade_date, total_value))
        cur_trade_date = get_before_after_trade_days(cur_trade_date, 1, is_before=False)
        left_trade_date = left_trade_date - 1
    print(holding_bond)
    return result

def get_result_value(result):
    dates = []
    value_all = []
    for val in result:
        dates.append(val[0])
        value_all.append(val[1])
    return dates, value_all

def plot_backtest_result(result_map):
    data_num = 10
    plt.figure(dpi=180)
    for key in result_map.keys():
        result = result_map[key]
        dates, value_all = get_result_value(result)
        date_num = len(dates)
        #print('tag:{}|date:{}|val:{}'.format(key, dates, value_all))
        plt.plot(dates, value_all, label=key)
    plt.legend()
    plt.xlabel('date')
    multiplier = int(date_num / data_num)
    if(multiplier > 1):
        plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(multiplier))
    plt.ylabel('value')
    plt.title('backtest')
    plt.gcf().autofmt_xdate()
    plt.show()

def show_backtest_result(start_date, days):
    trading_dates = days
    result = backtest(start_date=start_date, trading_dates=trading_dates, select_target_func=get_target_conv_bond_by_double_low, money=10000, candidate_bond_num=10)
    benchmark_result = backtest(start_date=start_date, trading_dates=trading_dates, select_target_func=get_benchmark_conv_bond, money=10000, candidate_bond_num=10)
    result_lst = {'double_low':result, 'benchmark':benchmark_result}
    plot_backtest_result(result_lst)

In [12]:
result=get_bond_info('2022-04-15')

total_df_len:451
process:123015|percentage:0.0
date:2022-04-15|bond:123015|stop trading
process:123021|percentage:0.0022172949002217295
date:2022-04-15|bond:123021|stop trading
process:123005|percentage:0.004434589800443459
date:2022-04-15|bond:123005|stop trading
process:123003|percentage:0.0066518847006651885
date:2022-04-15|bond:123003|stop trading
process:123002|percentage:0.008869179600886918
date:2022-04-15|bond:123002|stop trading
process:123004|percentage:0.011086474501108648
date:2022-04-15|bond:123004|stop trading
process:123006|percentage:0.013303769401330377
date:2022-04-15|bond:123006|stop trading
process:123016|percentage:0.015521064301552107
date:2022-04-15|bond:123016|stop trading
process:123008|percentage:0.017738359201773836
date:2022-04-15|bond:123008|stop trading
process:123013|percentage:0.019955654101995565
date:2022-04-15|bond:123013|stop trading
process:123010|percentage:0.022172949002217297
date:2022-04-15|bond:123010|stop trading
process:123011|percentage:0.02

date:2022-04-15|bond:128056|stop trading
process:113021|percentage:0.21286031042128603
date:2022-04-15|bond:113021|stop trading
process:110052|percentage:0.21507760532150777
process:113529|percentage:0.21729490022172948
date:2022-04-15|bond:113529|stop trading
process:110053|percentage:0.21951219512195122
date:2022-04-15|bond:110053|stop trading
process:110055|percentage:0.22172949002217296
process:110054|percentage:0.22394678492239467
date:2022-04-15|bond:110054|stop trading
process:110056|percentage:0.2261640798226164
date:2022-04-15|bond:110056|stop trading
process:128060|percentage:0.22838137472283815
date:2022-04-15|bond:128060|stop trading
process:113530|percentage:0.23059866962305986
date:2022-04-15|bond:113530|stop trading
process:113531|percentage:0.2328159645232816
date:2022-04-15|bond:113531|stop trading
process:110057|percentage:0.23503325942350334
date:2022-04-15|bond:110057|stop trading
process:128062|percentage:0.23725055432372505
date:2022-04-15|bond:128062|stop trading

date:2022-04-15|bond:128108|stop trading
process:127017|percentage:0.43458980044345896
date:2022-04-15|bond:127017|stop trading
process:128109|percentage:0.43680709534368073
date:2022-04-15|bond:128109|stop trading
process:123052|percentage:0.43902439024390244
date:2022-04-15|bond:123052|stop trading
process:113584|percentage:0.44124168514412415
date:2022-04-15|bond:113584|stop trading
process:113585|percentage:0.4434589800443459
process:123054|percentage:0.44567627494456763
date:2022-04-15|bond:123054|stop trading
process:128111|percentage:0.44789356984478934
date:2022-04-15|bond:128111|stop trading
process:128114|percentage:0.4501108647450111
date:2022-04-15|bond:128114|stop trading
process:113588|percentage:0.4523281596452328
date:2022-04-15|bond:113588|stop trading
process:113589|percentage:0.45454545454545453
date:2022-04-15|bond:113589|stop trading
process:123056|percentage:0.4567627494456763
date:2022-04-15|bond:123056|stop trading
process:128116|percentage:0.458980044345898
dat

date:2022-04-15|bond:128142|stop trading
process:123089|percentage:0.6541019955654102
date:2022-04-15|bond:123089|stop trading
process:123090|percentage:0.656319290465632
date:2022-04-15|bond:123090|stop trading
process:113615|percentage:0.6585365853658537
process:123091|percentage:0.6607538802660754
date:2022-04-15|bond:123091|stop trading
process:123092|percentage:0.6629711751662971
date:2022-04-15|bond:123092|stop trading
process:113616|percentage:0.6651884700665188
date:2022-04-15|bond:113616|stop trading
process:127028|percentage:0.6674057649667405
date:2022-04-15|bond:127028|stop trading
process:128143|percentage:0.6696230598669624
date:2022-04-15|bond:128143|stop trading
process:113618|percentage:0.6718403547671841
date:2022-04-15|bond:113618|stop trading
process:123093|percentage:0.6740576496674058
date:2022-04-15|bond:123093|stop trading
process:113619|percentage:0.6762749445676275
date:2022-04-15|bond:113619|stop trading
process:123095|percentage:0.6784922394678492
exception 

process:111002|percentage:0.8913525498891353
process:113635|percentage:0.893569844789357
process:113636|percentage:0.8957871396895787
process:123132|percentage:0.8980044345898004
process:113637|percentage:0.9002217294900222
process:123133|percentage:0.9024390243902439
process:127052|percentage:0.9046563192904656
process:110084|percentage:0.9068736141906873
process:113052|percentage:0.9090909090909091
process:123134|percentage:0.9113082039911308
process:123135|percentage:0.9135254988913526
process:113638|percentage:0.9157427937915743
process:118004|percentage:0.917960088691796
process:113053|percentage:0.9201773835920177
process:123136|percentage:0.9223946784922394
process:113639|percentage:0.9246119733924612
process:127053|percentage:0.926829268292683
process:118005|percentage:0.9290465631929047
process:123137|percentage:0.9312638580931264
process:127054|percentage:0.9334811529933481
process:113640|percentage:0.9356984478935698
process:127055|percentage:0.9379157427937915
process:11364

In [ ]:
# 拉取所需数据，以dataframe保存成文件，方便回测使用
# 一天数据需6min左右，请耐心等待
date = '2020-09-22'
end_date = '2022-04-09'
cur_date = date
while(cur_date < end_date):
    get_bond_info(cur_date)
    cur_date = get_before_after_trade_days(cur_date, 1, is_before=False)

total_df_len:565
process:123015|percentage:0.0
exception exist for code:123015|except_total_num:1
process:123021|percentage:0.0017699115044247787
exception exist for code:123021|except_total_num:2
process:123017|percentage:0.0035398230088495575
exception exist for code:123017|except_total_num:3
process:123005|percentage:0.005309734513274336
exception exist for code:123005|except_total_num:4
process:123003|percentage:0.007079646017699115
exception exist for code:123003|except_total_num:5
process:123002|percentage:0.008849557522123894
exception exist for code:123002|except_total_num:6
process:123004|percentage:0.010619469026548672
exception exist for code:123004|except_total_num:7
process:123006|percentage:0.012389380530973451
exception exist for code:123006|except_total_num:8
process:123016|percentage:0.01415929203539823
exception exist for code:123016|except_total_num:9
process:123007|percentage:0.01592920353982301
exception exist for code:123007|except_total_num:10
process:123008|perc

exception exist for code:128038|except_total_num:88
process:113506|percentage:0.15575221238938053
exception exist for code:113506|except_total_num:89
process:113507|percentage:0.15752212389380532
exception exist for code:113507|except_total_num:90
process:113508|percentage:0.1592920353982301
exception exist for code:113508|except_total_num:91
process:113509|percentage:0.16106194690265488
exception exist for code:113509|except_total_num:92
process:128039|percentage:0.16283185840707964
exception exist for code:128039|except_total_num:93
process:128040|percentage:0.16460176991150444
exception exist for code:128040|except_total_num:94
process:113510|percentage:0.1663716814159292
exception exist for code:113510|except_total_num:95
process:113511|percentage:0.168141592920354
exception exist for code:113511|except_total_num:96
process:110044|percentage:0.16991150442477876
exception exist for code:110044|except_total_num:97
process:113512|percentage:0.17168141592920355
exception exist for code

process:113543|percentage:0.30442477876106194
exception exist for code:113543|except_total_num:173
process:128071|percentage:0.30619469026548674
exception exist for code:128071|except_total_num:174
process:123029|percentage:0.30796460176991153
exception exist for code:123029|except_total_num:175
process:123030|percentage:0.30973451327433627
exception exist for code:123030|except_total_num:176
process:128072|percentage:0.31150442477876106
exception exist for code:128072|except_total_num:177
process:128073|percentage:0.31327433628318585
exception exist for code:128073|except_total_num:178
process:123031|percentage:0.31504424778761064
exception exist for code:123031|except_total_num:179
process:128074|percentage:0.3168141592920354
exception exist for code:128074|except_total_num:180
process:128075|percentage:0.3185840707964602
exception exist for code:128075|except_total_num:181
process:123032|percentage:0.32035398230088497
exception exist for code:123032|except_total_num:182
process:1280

exception exist for code:128105|except_total_num:260
process:113034|percentage:0.46017699115044247
exception exist for code:113034|except_total_num:261
process:128106|percentage:0.46194690265486726
exception exist for code:128106|except_total_num:262
process:123048|percentage:0.46371681415929206
exception exist for code:123048|except_total_num:263
process:113576|percentage:0.4654867256637168
exception exist for code:113576|except_total_num:264
process:123049|percentage:0.4672566371681416
exception exist for code:123049|except_total_num:265
process:110070|percentage:0.4690265486725664
exception exist for code:110070|except_total_num:266
process:123050|percentage:0.47079646017699117
exception exist for code:123050|except_total_num:267
process:113577|percentage:0.4725663716814159
exception exist for code:113577|except_total_num:268
process:113573|percentage:0.4743362831858407
exception exist for code:113573|except_total_num:269
process:113578|percentage:0.4761061946902655
exception exist 

process:128130|percentage:0.6088495575221239
exception exist for code:128130|except_total_num:345
process:123064|percentage:0.6106194690265486
exception exist for code:123064|except_total_num:346
process:123065|percentage:0.6123893805309735
exception exist for code:123065|except_total_num:347
process:128131|percentage:0.6141592920353982
exception exist for code:128131|except_total_num:348
process:128132|percentage:0.6159292035398231
exception exist for code:128132|except_total_num:349
process:123066|percentage:0.6176991150442478
exception exist for code:123066|except_total_num:350
process:123067|percentage:0.6194690265486725
exception exist for code:123067|except_total_num:351
process:128133|percentage:0.6212389380530974
exception exist for code:128133|except_total_num:352
process:113603|percentage:0.6230088495575221
exception exist for code:113603|except_total_num:353
process:128134|percentage:0.6247787610619469
exception exist for code:128134|except_total_num:354
process:113604|perce

exception exist for code:110808|except_total_num:430
process:113620|percentage:0.7610619469026548
exception exist for code:113620|except_total_num:431
process:123101|percentage:0.7628318584070797
exception exist for code:123101|except_total_num:432
process:123103|percentage:0.7646017699115044
exception exist for code:123103|except_total_num:433
process:123102|percentage:0.7663716814159292
exception exist for code:123102|except_total_num:434
process:123104|percentage:0.768141592920354
exception exist for code:123104|except_total_num:435
process:123105|percentage:0.7699115044247787
exception exist for code:123105|except_total_num:436
process:127029|percentage:0.7716814159292036
exception exist for code:127029|except_total_num:437
process:113046|percentage:0.7734513274336283
exception exist for code:113046|except_total_num:438
process:127030|percentage:0.7752212389380531
exception exist for code:127030|except_total_num:439
process:128145|percentage:0.7769911504424779
exception exist for c

In [ ]:
# 展示从2020-09-22开始一年的收益率
show_backtest_result('2020-09-22', 240)